# 02 — Compute Graph Structural Properties

This notebook computes structural graph properties for all Max-Cut
instances used in the benchmark dataset.

The extracted properties are used for downstream analyses examining
the relationship between graph structure and solver performance.

Computed properties include:

- bipartiteness
- triangle statistics
- clustering metrics
- bridge statistics
- cyclomatic number
- negative edge statistics
- unique edge weights

In [8]:
# Imports
import sys
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

from tqdm import tqdm


# Project Imports
sys.path.append(str(Path("../src/maxcut").resolve()))

%load_ext autoreload
%autoreload 2

import config as cfg
import utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# Paths
PATH_BASELINE = (
    cfg.DATA_DIR / "metadata/baseline.csv"
)

INST_ROOT = (
    cfg.RAW_DIR / "mqlib_instances"
)

OUTPUT_PATH = (
    cfg.PROCESSED_DIR / "graph_metrics.csv"
)
ERROR_LOG = (
    cfg.PROCESSED_DIR / "graph_metric_errors.log"
)

In [10]:
# Load Baseline Metadata
df_baseline = pd.read_csv(PATH_BASELINE)

print(f"Instances: {len(df_baseline):,}")

df_baseline.head()

Instances: 3,506


,name,nodes,edges,density,limit,has_leafs,int_only
0,G55,4969,12498,0.001013,13.945,True,True
1,G56,4969,12498,0.001013,13.922,True,True
2,G57,5000,10000,0.000800,11.299,False,True
3,G58,5000,29570,0.002366,11.876,False,True
4,G59,5000,29570,0.002366,15.875,False,True


In [6]:
# Compute Structural Metrics

def compute_graph_metrics(name):

    zip_path = utils.find_zip_for_graph(name)

    if zip_path is None:
        return None

    n, edges = utils.read_edgelist_from_zip(zip_path)

    G = nx.Graph()

    G.add_nodes_from(range(n))
    G.add_weighted_edges_from(edges)

    n_nodes = G.number_of_nodes()
    m_edges = G.number_of_edges()

    weights = [w for (_, _, w) in edges]

    neg_w = [w for w in weights if w < 0]


    # Triangles 
    tri_dict = nx.triangles(G)

    total_triangles = (
        sum(tri_dict.values()) // 3
    )

   
    # Bridges
    bridges = list(nx.bridges(G))

    
    # Cyclomatic Number
    components = nx.number_connected_components(G)

    cyclomatic = (
        m_edges - n_nodes + components
    )

   
    # Return Metrics
    return {

        "name": name,

        # Graph Size
        "nodes": n_nodes,
        "edges": m_edges,

        # Bipartite
        "bipartite": (
            nx.is_bipartite(G)
            if n_nodes > 0
            else False
        ),

        # Negative Edges
        "neg_edges": len(neg_w),

        "neg_edges_frac": (
            len(neg_w) / m_edges
            if m_edges > 0 else 0
        ),

        "neg_wt_sum": sum(neg_w),

        "neg_wt_ratio": (
            sum(abs(w) for w in neg_w)
            /
            sum(abs(w) for w in weights)
            if weights else 0
        ),

        # Triangles
        "triangle_count": total_triangles,

        "has_triangles": (
            total_triangles > 0
        ),

        # Clustering
        "transitivity": nx.transitivity(G),

        "average_clustering": (
            nx.average_clustering(G)
        ),

        # Cycles
        "cyclomatic_number": cyclomatic,

        # Bridges
        "bridge_count": len(bridges),

        "bridge_fraction": (
            len(bridges) / m_edges
            if m_edges > 0 else 0
        ),

        # Unique Weights
        "unique_weights": (
            len(set(weights))
        )
    }

In [ ]:
if OUTPUT_PATH.exists():

    df_existing = pd.read_csv(OUTPUT_PATH)

    processed = set(df_existing["name"])

    print(f"Found existing file.")
    print(f"Already processed: {len(processed)}")

else:

    processed = set()


results_buffer = []

SAVE_EVERY = 10

for name in tqdm(df_baseline["name"]):

    # Skip completed graphs
    if name in processed:
        continue

    try:

        metrics = compute_graph_metrics(name)

        if metrics is not None:

            results_buffer.append(metrics)

        # ------------------------------------------------
        # Periodic checkpoint save
        # ------------------------------------------------
        if len(results_buffer) >= SAVE_EVERY:

            df_chunk = pd.DataFrame(results_buffer)

            if OUTPUT_PATH.exists():

                df_chunk.to_csv(
                    OUTPUT_PATH,
                    mode="a",
                    header=False,
                    index=False
                )

            else:

                df_chunk.to_csv(
                    OUTPUT_PATH,
                    index=False
                )

            results_buffer = []

    except Exception as e:

        print(f"Error processing {name}: {e}")

        with open(ERROR_LOG, "a") as f:
            f.write(f"{name}: {e}\n")

# ---------------------------------------------------
# Save remaining rows
# ---------------------------------------------------

if results_buffer:

    df_chunk = pd.DataFrame(results_buffer)

    if OUTPUT_PATH.exists():

        df_chunk.to_csv(
            OUTPUT_PATH,
            mode="a",
            header=False,
            index=False
        )

    else:

        df_chunk.to_csv(
            OUTPUT_PATH,
            index=False
        )

print("Finished.")